In [2]:
import tensorflow as tf
import numpy as np

# Load MNIST Dataset
def load_mnist():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
    x_train, x_test = x_train / 255.0, x_test / 255.0  # Normalize to [0, 1]
    x_train = np.expand_dims(x_train, axis=-1)  # Add channel dimension
    x_test = np.expand_dims(x_test, axis=-1)
    return (x_train, y_train), (x_test, y_test)

# Generate Rotated Images and Labels for the Pretext Task
def generate_rotated_data(images):
    rotations = [0, 90, 180, 270]
    rotated_images = []
    rotation_labels = []
    for image in images:
        for i, angle in enumerate(rotations):
            rotated_image = tf.image.rot90(image, k=i)  # Rotate image by 90° * k
            rotated_images.append(rotated_image)
            rotation_labels.append(i)  # Label corresponds to the rotation index
    return np.array(rotated_images), np.array(rotation_labels)

# Build the Rotation Prediction Model
def build_rotation_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=input_shape),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(4, activation='softmax')  # 4 classes: 0°, 90°, 180°, 270°
    ])
    return model

# Train the Rotation Pretext Task
def train_rotation_task(x_train):
    # Generate rotated images and labels
    rotated_images, rotation_labels = generate_rotated_data(x_train)

    # Build and compile the model
    model = build_rotation_model(input_shape=x_train.shape[1:])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Train the model
    model.fit(rotated_images, rotation_labels, epochs=5, batch_size=64, validation_split=0.1)

    # Return the trained model
    return model

# Compute Prototypes for Few-Shot Learning
# This function computes the mean embedding for each class in the support set
def compute_prototypes(encoder, x_support, y_support):
    embeddings = encoder.predict(x_support)  # Extract embeddings using the encoder
    prototypes = {}
    for label in np.unique(y_support):  # Iterate over unique class labels
        prototypes[label] = embeddings[y_support == label].mean(axis=0)  # Compute mean embedding
    return prototypes

# Classify Query Samples Using Prototypes
# This function classifies query samples based on the closest prototype
def classify_query(encoder, prototypes, x_query):
    query_embeddings = encoder.predict(x_query)  # Extract embeddings for query samples
    predictions = []
    for query in query_embeddings:
        # Compute distances to all prototypes
        distances = {label: np.linalg.norm(query - proto) for label, proto in prototypes.items()}
        # Assign the label of the closest prototype
        predictions.append(min(distances, key=distances.get))
    return predictions

In [4]:
import gym
from gym import spaces

# Reinforcement Learning: Augmentation Environment
class AugmentationEnv(gym.Env):
    def __init__(self, model, x_train, y_train):
        super(AugmentationEnv, self).__init__()
        self.model = model
        self.x_train = x_train
        self.y_train = y_train
        self.action_space = spaces.Discrete(3)  # Example: 3 augmentations
        self.observation_space = spaces.Box(low=0, high=1, shape=x_train[0].shape, dtype=np.float32)
        self.current_index = 0

    def step(self, action):
        # Apply augmentation based on action
        augmented_image = self._apply_augmentation(self.x_train[self.current_index], action)
        reward = self._compute_reward(augmented_image, self.y_train[self.current_index])
        self.current_index = (self.current_index + 1) % len(self.x_train)
        return augmented_image, reward, False, {}

    def reset(self):
        self.current_index = 0
        return self.x_train[self.current_index]

    def _apply_augmentation(self, image, action):
        if action == 0:  # No augmentation
            return image
        elif action == 1:  # Rotate
            return tf.image.rot90(image)
        elif action == 2:  # Flip
            return tf.image.flip_left_right(image)

    def _compute_reward(self, augmented_image, label):
        # Compute reward based on model performance
        prediction = np.argmax(self.model.predict(np.expand_dims(augmented_image, axis=0)))
        return 1.0 if prediction == label else -1.0

In [11]:
# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = load_mnist()

# Step 1: Self-Supervised Learning with Rotation Pretext Task
rotation_model = train_rotation_task(x_train)


KeyboardInterrupt: 

In [10]:
# Fix: Build the rotation model by calling it with dummy input
dummy_input = np.expand_dims(x_train[0], axis=0)  # Add batch dimension
rotation_model.predict(dummy_input)

# Step 2: Few-Shot Learning
# Use the encoder part of the rotation model for few-shot learning
encoder = tf.keras.Model(inputs=rotation_model.input, outputs=rotation_model.layers[-2].output)

# Few-shot learning setup
x_support, y_support = x_train[:100], y_train[:100]
x_query, y_query = x_train[100:200], y_train[100:200]
prototypes = compute_prototypes(encoder, x_support, y_support)
predictions = classify_query(encoder, prototypes, x_query)
accuracy = np.mean(np.array(predictions) == y_query)
print(f"Few-Shot Learning Accuracy: {accuracy:.2f}")

# Step 3: Reinforcement Learning
# Use the encoder as the model for RL
env = AugmentationEnv(model=encoder, x_train=x_train, y_train=y_train)
state = env.reset()
for _ in range(10):  # Example: 10 steps
    action = env.action_space.sample()  # Random action
    next_state, reward, done, _ = env.step(action)
    print(f"Action: {action}, Reward: {reward}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step


AttributeError: The layer sequential_1 has never been called and thus has no defined input.